# 第6周 · 课堂练习：泄漏排查演练

先做一个数值梯度热身，然后进入主戏：一份"看起来很顺、结果好得吓人"的流水线，里面埋了 **3 处数据泄漏**，限时找出并修复。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week06 as ck

## E1 · 热身：数值梯度

重写课上的 `numerical_gradient(f, x)`（中心差分），并验证 $g(x) = x^3 + 2x$ 在 $x = 2$ 处的导数（先手推解析值：$g'(x) = 3x^2 + 2$）。

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    # 在这里写你的代码
    raise NotImplementedError

# 在这里写你的代码
raise NotImplementedError
print("解析:", analytic, "  数值:", numeric)
ck.numerical_gradient(numerical_gradient)
ck.analytic_derivative(analytic)

## E2 · 埋雷排查：一位师兄的 IMDB 流水线

一位师兄发来他的情感分类流水线，兴奋地说测试准确率接近满分，"下周投稿"。代码确实能跑、也没有语法错误。**但真实水平的流水线在这份数据上只能拿到 0.87 左右。**

你的任务：

1. 运行下面两格，确认那个"好得吓人"的数字；
2. **逐行读代码**（数据准备那一格也是可疑区域！），找出 3 处数据泄漏；
3. 修复它们，报告修复前后的测试准确率。

背景知识（都是真事）：原始 IMDB 数据集的 train 与 test 之间有 **123 条一字不差的重复影评**；原始数据文件**按标签排序存放**（前一半全是差评）。

In [ ]:
# ---------- 师兄的数据准备（这一格也要认真读！） ----------
rng = np.random.default_rng(2026)
train_df = data.load_imdb("train")
test_df = data.load_imdb("test")

# "测试集有几条标注可疑，我用训练集里质量高的影评补齐了 300 条"
src = rng.choice(len(train_df), size=300, replace=False)
dst = rng.choice(len(test_df), size=300, replace=False)
keep = np.setdiff1d(np.arange(len(test_df)), dst)
test_df = pd.concat([test_df.iloc[keep], train_df.iloc[src][["text", "label"]]],
                    ignore_index=True)

# "按标签排个序、给每行一个编号，方便之后核对"
train_df = train_df.sort_values("label", kind="stable").reset_index(drop=True)
test_df = test_df.sort_values("label", kind="stable").reset_index(drop=True)
train_df["sample_id"] = np.arange(len(train_df))
test_df["sample_id"] = np.arange(len(test_df))
train_df.head(3)

In [ ]:
# ---------- 师兄的模型流水线 ----------
import scipy.sparse as sp          # scipy 随 scikit-learn 一起安装，这里只用它拼接稀疏矩阵
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# "为了让词表更全，向量化器在全部数据上拟合"
vec = TfidfVectorizer(min_df=2)
vec.fit(pd.concat([train_df["text"], test_df["text"]]))


def make_features(df):
    """TF-IDF 特征 + 归一化的样本编号（'编号也许包含有用信息'）"""
    ids = (df["sample_id"] / len(df)).to_numpy().reshape(-1, 1)
    return sp.hstack([vec.transform(df["text"]), sp.csr_matrix(ids)])


clf = LogisticRegression(max_iter=1000).fit(make_features(train_df), train_df["label"])
leaky_acc = clf.score(make_features(test_df), test_df["label"])
print(f"测试准确率：{leaky_acc:.4f} —— 师兄已经在写摘要了")
ck.leak_inflated_accuracy(leaky_acc)

**任务 1**：写出你找到的 3 处泄漏（每处一两句：它在哪一行、为什么是泄漏）。

（在这里回答）

**任务 2**：修复你找到的全部泄漏，重新训练并报告修复后的测试准确率 `fixed_acc`。

为了让自查能逐项核对，请把修复后流水线的三个部件分别存进：`test_ok`（最终用来评估的测试集，DataFrame）、`vec_ok`（向量化器）、`clf_ok`（分类器）。约定：训练集 `train_df` 的 5000 条样本保持不变，评估只在给定测试集的样本里进行。自查会分别检查每一处修复——没通过时只给方向性提示，同一项第二次没通过时再给一个诊断实验。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"修复后的测试准确率：{fixed_acc:.4f}   （评估用的测试样本：{len(test_ok)} 条）")
ck.leak_fixed(test_ok, vec_ok, clf_ok, fixed_acc)

**任务 3**（讨论）：三处泄漏哪处"贡献"最大？你会设计什么实验来验证？如果师兄已经把 0.99 写进了论文草稿，你会怎么跟他说？

（在这里回答）